In [1]:
import numpy as np
foo = [1,2,3,4,5]
print(np.cumsum(foo))

[ 1  3  6 10 15]


In [30]:
import numpy as np
from scipy import integrate
from scipy.special import erf


def discretize_tilted_ellipse_IWC(width, A, angle_deg,
                                         total_ice_mass_kg, X,
                                         thickness_m=1.0):
    """
    Discretize a Gaussian IWC distribution over a TILTED ellipse into X
    vertical slices, given a TOTAL ICE MASS to distribute, with the
    Gaussian's peak amplitude normalized against its TRUE TRUNCATED
    integral (i.e. integrated only within the ellipse boundary, not over
    the full plane). This makes the slice masses sum to EXACTLY
    total_ice_mass_kg (to floating point precision) -- there is no
    "erf(sqrt(2)) = 95.45%" leftover gap like the y-unbounded approximation
    used previously, at the cost of needing one cheap 1D numerical
    integral per slice (no full 2D quadrature).

    SHAPE: standard 2D Gaussian in the ellipse's local (rotated) frame,
        G(u,v) = exp(-u^2/(2*sigma_a^2)) * exp(-v^2/(2*sigma_b^2))
    with sigma_a = a/2, sigma_b = b/2 (2-sigma = semi-axis length, so the
    ellipse boundary coincides with the Gaussian's 2-sigma contour).

    TRUNCATION FACTOR (closed form, angle-independent): because the
    ellipse boundary is a circle of radius 2 in standardized coordinates
    (u/sigma_a, v/sigma_b), the fraction of the untruncated Gaussian mass
    that lies within the ellipse is exactly (1 - exp(-2)) ~= 86.47%
    (equivalently, the CDF of a chi-squared distribution with 2 d.o.f. at
    x=4). This fraction is independent of rotation angle and aspect ratio,
    which is what makes exact normalization against the truncated total
    possible in closed form (no need to numerically integrate the total).

    MASS CONSERVATION: slice_mass_kg.sum() == total_ice_mass_kg to
    floating-point precision (verified numerically), since the peak
    amplitude is normalized against the EXACT truncated total, not an
    untruncated approximation.

    SPATIAL MEAN: as established, with mean defined as
        mean_IWC = (integral of IWC over ellipse) / A
    this is automatically satisfied once mass is conserved -- it's an
    algebraic consequence of the mass constraint, not a separate condition
    on shape.

    DIMENSIONAL HANDLING: same as the parabolic-cap version -- assumes an
    implicit slab thickness thickness_m (default 1 m) bridges IWC's
    volumetric units (g/m^3) to the 2D ellipse area (m^2):
        total_ice_mass_kg = (integral of IWC over ellipse area)
                             * thickness_m / 1000

    PERFORMANCE: each slice integral uses a closed-form (erf-based) inner
    integral over y (at fixed x, bounded by the true ellipse), wrapped in
    a single 1D `scipy.integrate.quad` over x. This is ~6x faster per call
    than full 2D `dblquad` in benchmarking, since one of the two
    quadrature dimensions has been collapsed into an exact closed form.
    There is no way to remove the remaining 1D quadrature in closed form,
    because the closed-form inner integral's dependence on x is not
    elementary (x enters inside erf arguments in a coupled way that does
    not have an antiderivative in elementary functions). If this 1D quad
    is still too slow for your use case (e.g. very large X or very many
    repeated calls), consider precomputing a fine-resolution lookup table
    of the inner integral as a function of x and using cumulative
    trapezoidal integration instead of repeated `quad` calls.

    Parameters:
        width             : total x-extent of the UNROTATED ellipse (= 2*a)
        A                 : cross-sectional area of the ellipse (m^2)
        D                 : effective depth = half-height at x=centroid
                             (currently unused in the area/semi-axis solve;
                             retained for interface compatibility)
        angle_deg         : angle of major axis from x-axis (degrees)
        total_ice_mass_kg : total ice mass to distribute over the ellipse (kg)
        X                 : number of vertical slices
        thickness_m       : implicit slab thickness (m), default 1.0

    Returns:
        slice_centers   : x positions of slice centers
        slice_iwc       : average IWC in each slice (g/m^3)
        slice_mass_kg   : ice mass (kg) integrated within each slice
                           (sums to total_ice_mass_kg to floating-point
                           precision)
    """
    alpha = np.radians(angle_deg)
    ca, sa = np.cos(alpha), np.sin(alpha)

    # -------------------------------------------------------------------
    # Step 1: Recover semi-axes a, b from (width, A, alpha)
    #
    # NOTE: as flagged previously, this treats `width` as the UNROTATED
    # extent (2*a). Revisit if `width` is meant to be the tilted
    # bounding-box extent instead -- independent of the mass/shape logic.
    # -------------------------------------------------------------------
    a0 = 0.5 * width / np.cos(alpha)
    b0 = A / (np.pi * a0)

    sigma_a = a0 / 2
    sigma_b = b0 / 2

    # -------------------------------------------------------------------
    # Step 2: x-extent of the tilted ellipse (per the original convention)
    # -------------------------------------------------------------------
    x_max = width / 2
    x_min = -x_max

    # True x-extent of the tilted ellipse (needed to clip slice bounds so
    # the closed-form inner integral's sqrt/erf arguments stay valid)
    K = a0**2 * ca**2 + b0**2 * sa**2
    x_extent_true = np.sqrt(K)

    # -------------------------------------------------------------------
    # Step 3: Peak IWC (g/m^3) from total ice mass (kg), normalized
    # against the EXACT TRUNCATED total (closed form, angle-independent):
    #   truncated_total = 2*pi*sigma_a*sigma_b * (1 - exp(-2))
    # -------------------------------------------------------------------
    total_ice_mass_g = total_ice_mass_kg * 1000.0
    truncated_total = 2.0 * np.pi * sigma_a * sigma_b * (1.0 - np.exp(-2.0))
    peak_iwc = total_ice_mass_g / (thickness_m * truncated_total)

    # -------------------------------------------------------------------
    # Step 4: Closed-form inner (y-bounded) integral at fixed x, using the
    # TRUE ellipse y-limits (not unbounded y). Derived by completing the
    # square in y for the rotated-frame Gaussian; verified against direct
    # 1D quadrature to machine precision.
    # -------------------------------------------------------------------
    def y_limits(x):
        A_q = sa**2 / a0**2 + ca**2 / b0**2
        B_q = 2 * x * ca * sa * (1 / a0**2 - 1 / b0**2)
        C_q = x**2 * (ca**2 / a0**2 + sa**2 / b0**2) - 1
        disc = B_q**2 - 4 * A_q * C_q
        if disc < 0:
            return None, None
        sqrt_disc = np.sqrt(disc)
        y1 = (-B_q - sqrt_disc) / (2 * A_q)
        y2 = (-B_q + sqrt_disc) / (2 * A_q)
        return min(y1, y2), max(y1, y2)

    P = sa**2 / (2 * sigma_a**2) + ca**2 / (2 * sigma_b**2)

    def inner_integral(x):
        ylo, yhi = y_limits(x)
        if ylo is None:
            return 0.0
        Q = -ca * sa * x / sigma_b**2 + ca * sa * x / sigma_a**2
        R = ca**2 * x**2 / (2 * sigma_a**2) + sa**2 * x**2 / (2 * sigma_b**2)
        y0 = -Q / (2 * P)
        prefactor = np.exp(-(R - Q**2 / (4 * P)))
        return prefactor * np.sqrt(np.pi / (4 * P)) * (
            erf(np.sqrt(P) * (yhi - y0)) - erf(np.sqrt(P) * (ylo - y0))
        )

    # -------------------------------------------------------------------
    # Step 5: Build slices. Each slice integral is a single 1D `quad` over
    # x of the closed-form inner_integral (no 2D quadrature).
    # -------------------------------------------------------------------
    dx = width / X
    slice_centers = np.linspace(x_min + dx / 2, x_max - dx / 2, X)

    slice_mass_g = np.zeros(X)
    slice_iwc = np.zeros(X)

    for i, x_c in enumerate(slice_centers):
        x_lo = np.clip(x_c - dx / 2, -x_extent_true, x_extent_true)
        x_hi = np.clip(x_c + dx / 2, -x_extent_true, x_extent_true)

        if x_hi <= x_lo:
            slice_mass_g[i] = 0.0
            slice_iwc[i] = 0.0
            continue

        unscaled_mass, _ = integrate.quad(inner_integral, x_lo, x_hi)
        slice_mass_g[i] = peak_iwc * unscaled_mass

        y_half = _ellipse_y_half(x_c, a0, b0, ca, sa)
        slice_width = x_hi - x_lo
        slice_area = slice_width * 2 * y_half if y_half is not None else np.nan
        slice_iwc[i] = slice_mass_g[i] / slice_area if slice_area else 0.0

    slice_mass_kg = slice_mass_g / 1000.0

    return slice_centers, slice_iwc, slice_mass_kg


def _ellipse_y_half(x, a0, b0, ca, sa):
    """Half-height (max |y|) of the tilted ellipse boundary at position x."""
    A_q = sa**2 / a0**2 + ca**2 / b0**2
    B_q = 2 * x * ca * sa * (1 / a0**2 - 1 / b0**2)
    C_q = x**2 * (ca**2 / a0**2 + sa**2 / b0**2) - 1
    disc = B_q**2 - 4 * A_q * C_q
    if disc < 0:
        return None
    sqrt_disc = np.sqrt(disc)
    y1 = (-B_q - sqrt_disc) / (2 * A_q)
    y2 = (-B_q + sqrt_disc) / (2 * A_q)
    return (max(y1, y2) - min(y1, y2)) / 2.0

In [28]:
def discretize_tilted_ellipse(width, A, D, angle_deg, bulk_property, X):
    """
    Discretize a Gaussian mass distribution over a TILTED ellipse into X vertical slices.

    The Gaussian is defined in the ellipse's local frame (along semi-axes a, b).
    The 2-sigma contour of the Gaussian coincides with the ellipse boundary.

    Parameters:
        width      : total x-extent of the tilted ellipse
        A          : cross-sectional area of the ellipse
        D          : effective depth = half-height at x=centroid (x=0)
        angle_deg  : angle of major axis from x-axis (degrees)
        bulk_property : bulk property to distribute (e.g., mass, number)
        X          : number of vertical slices

    Returns:
        slice_centers : x positions of slice centers
        slice_masses  : integrated mass in each slice
    """
    alpha = np.radians(angle_deg)
    ca, sa = np.cos(alpha), np.sin(alpha)

    # -------------------------------------------------------------------------
    # Step 1: Recover semi-axes a, b from (width, D, alpha)
    # -------------------------------------------------------------------------

    a0 = 0.5*width / np.cos(alpha)
    b0 = A / (np.pi * a0)

    print("Initial guess for semi-axes (a0, b0): ", a0, b0)
    print("Angle alpha (degrees): ", angle_deg)

    A_check = np.pi * a0 * b0
    print(f"Semi-major axis a : {a0:.4f}")
    print(f"Semi-minor axis b : {b0:.4f}")
    print(f"Target area       : {A:.4f}")
    print(f"Recovered area    : {A_check:.4f}  (error: {abs(A_check-A):.2e})")

    # Gaussian sigmas in the ellipse's local frame (2-sigma = semi-axis length)
    sigma_a = a0 / 2
    sigma_b = b0 / 2

    # -------------------------------------------------------------------------
    # Step 2: x-extent of the tilted ellipse
    # -------------------------------------------------------------------------
    x_max = width / 2   # = sqrt((a*ca)^2 + (b*sa)^2), consistent by construction
    x_min = -x_max

    # -------------------------------------------------------------------------
    # Step 3: y-limits at a given x by intersecting vertical line with tilted ellipse
    #
    # Substitute rotated coords u = x*ca + y*sa, v = -x*sa + y*ca into
    # (u/a)^2 + (v/b)^2 = 1  =>  quadratic in y:
    #   (sa^2/a^2 + ca^2/b^2)*y^2
    #   + 2*x*(ca*sa/a^2 - ca*sa/b^2)*y
    #   + x^2*(ca^2/a^2 + sa^2/b^2) - 1 = 0
    # -------------------------------------------------------------------------
    def y_limits(x):
        A_q = sa**2 / a0**2 + ca**2 / b0**2
        B_q = 2 * x * ca * sa * (1 / a0**2 - 1 / b0**2)
        C_q = x**2 * (ca**2 / a0**2 + sa**2 / b0**2) - 1
        disc = B_q**2 - 4 * A_q * C_q
        if disc < 0:
            return None, None   # x is outside the ellipse
        sqrt_disc = np.sqrt(disc)
        y1 = (-B_q - sqrt_disc) / (2 * A_q)
        y2 = (-B_q + sqrt_disc) / (2 * A_q)
        return min(y1, y2), max(y1, y2)

    # -------------------------------------------------------------------------
    # Step 4: 2D Gaussian in the ellipse's LOCAL (rotated) frame
    #
    # u =  x*cos(a) + y*sin(a)   (along major axis)
    # v = -x*sin(a) + y*cos(a)   (along minor axis)
    # G(x,y) = exp(-u^2 / 2*sigma_a^2) * exp(-v^2 / 2*sigma_b^2)
    # -------------------------------------------------------------------------
    def gaussian_2d(x, y):
        u =  x * ca + y * sa
        v = -x * sa + y * ca
        return (np.exp(-u**2 / (2 * sigma_a**2)) *
                np.exp(-v**2 / (2 * sigma_b**2)))

    # -------------------------------------------------------------------------
    # Step 6: Integrate each vertical slice
    #
    # For each x, the y-integral of the Gaussian can use erf IF the Gaussian
    # were axis-aligned in y. But since G is in the rotated frame, the
    # y-dependence at fixed x mixes u and v — so we keep full numerical quad.
    # -------------------------------------------------------------------------
    scale = bulk_property

    dx = width / X
    slice_centers = np.linspace(x_min + dx / 2, x_max - dx / 2, X)
    slice_properties = np.zeros(X)

    mass_sum = np.zeros(X)

    for i, x_c in enumerate(slice_centers):
        x_lo = np.clip(x_c - dx / 2, x_min, x_max)
        x_hi = np.clip(x_c + dx / 2, x_min, x_max)

        def strip_integrand(y, x):
            return gaussian_2d(x, y)

        def y_lo_strip(x):
            lo, _ = y_limits(x)
            return lo if lo is not None else 0.0

        def y_hi_strip(x):
            _, hi = y_limits(x)
            return hi if hi is not None else 0.0

        mass_strip, _ = integrate.dblquad(
            strip_integrand,
            x_lo, x_hi,
            y_lo_strip, y_hi_strip
        )

        # compute slice area for averaging
        slice_area, _ = integrate.dblquad(
            lambda y, x: 1.0,
            x_lo, x_hi,
            y_lo_strip, y_hi_strip
        )

        slice_properties[i] = scale * mass_strip / slice_area
        mass_sum[i] = scale * mass_strip
    print(f"Total ice mass predicted is: {np.sum(mass_sum)}")
    return slice_properties

In [15]:
### libRadtran FUNCTION LIBRARY ###
import subprocess
import numpy as np
import warnings
from scipy.special import erf
from scipy import integrate, optimize

In [1]:
import glob
import matplotlib.pyplot as plt
import sys
import xarray as xr
import numpy as np
import pandas as pd
import re
from matplotlib.lines import Line2D
import os
from matplotlib import cm
import matplotlib.patches as mpatches
from matplotlib import ticker
from matplotlib.path import Path
import matplotlib.colors as mcolors
from matplotlib.patches import Ellipse
import matplotlib.gridspec as gridspec
import csv

sys.path.append('/home/chinahg/GCresearch/contrailuncertainty/start_here/')
import pipeline_fxn_lib as lib
sys.path.append('/home/chinahg/GCresearch/contrailuncertainty/LRT/')
import LRT_fxnlib as LRTlib

plt.rcParams["font.family"] = "FreeSerif"
plt.rcParams["mathtext.fontset"] = "cm"

temp_groups = ['205', '218', '225']
color_map = {
    '205': 'gray',
    '218': 'red',
    '225': 'green'
}

rho_ice = 917.0  # kg/m^3, used for CoCiP mass calculation
model_colors = {"APCEMM": "tab:blue", "LES": "tab:orange", "CoCiP": "tab:green"}
rh_styles = {110: "--", 130: "-"}

In [2]:
test_ids = ['110T205L25', '110T218L25', '110T225L25', '130T205L25', '130T218L25', '130T225L25']
hours = [0, 12]
habits = ['ghm', 'solid-column', 'rough-aggregate']
base_dir = '/home/chinahg/GCresearch/contrailuncertainty/LRT/RF_EF_results'

# APCEMM
apcemm_data = {
    (tid, hour, habit): xr.open_dataset(f'{base_dir}/APCEMM/{tid}/APCEMM_{tid}_{hour}h_{habit}.nc', decode_times=False)
    for tid in test_ids
    for hour in hours
    for habit in habits
}

apcemm_default_data = {
    (tid, hour, habit): xr.open_dataset(f'{base_dir}/APCEMM/default_settings/{tid}/APCEMM_{tid}_{hour}h_{habit}.nc', decode_times=False)
    for tid in test_ids
    for hour in hours
    for habit in habits
}

# CoCiP
cocip_data = {
    (tid, hour, habit): xr.open_dataset(f'{base_dir}/CoCiP/{tid}/CoCiP_{tid}_{hour}h_{habit}.nc', decode_times=False)
    for tid in test_ids
    for hour in hours
    for habit in habits
}

# LES
les_data = {
    (tid, hour, habit): xr.open_dataset(f'{base_dir}/LES/{tid}/LES_{tid}_{hour}h_{habit}.nc', decode_times=False)
    for tid in test_ids
    for hour in hours
    for habit in habits
}

In [4]:
key = ('110T218L25', 0, 'ghm')
cds = cocip_data[key]

In [33]:
width = cds['width'][10]
A = cds['area_eff'][10]
D = cds['depth'][10]
angle_deg = 0
bulk_property = cds['iwc'][10] * cds["rho_air"][10] * 1e3
X = 25

output = discretize_tilted_ellipse(width, A, D, angle_deg, bulk_property, X)

Initial guess for semi-axes (a0, b0):  <xarray.DataArray 'width' ()>
array(4408.53533785)
Coordinates:
    t        int64 10 <xarray.DataArray ()>
array(111.44215472)
Coordinates:
    t        int64 10
Angle alpha (degrees):  0
Semi-major axis a : 4408.5353
Semi-minor axis b : 111.4422
Target area       : 1543454.0319
Recovered area    : 1543454.0319  (error: 0.00e+00)
Total ice mass predicted is: 1325.241426531581


In [ ]:
actual_mass = cds['n_ice_per_m'][10] * 4/3 * np.pi * cds['r_ice_vol'][10]**3 * rho_ice * 1e6
print(f"actual mass: {actual_mass.values:.3} [kg]")

actual mass: 3.1e+06 [kg]


In [32]:
slice_centers, slice_iwc, slice_mass_kg = discretize_tilted_ellipse_IWC(width, A, angle_deg,
                                         actual_mass, X,
                                         thickness_m=1.0)
print(f"{np.sum(slice_mass_kg):.3} [kg]")

3.07e+06 [kg]
